In [ ]:
# Standard library
import os
import re
import warnings
from pathlib import Path
from collections import Counter, defaultdict

# Third-party scientific stack
import pandas as pd
import numpy as np

# Scientific computing
import scipy.stats as stats
from scipy.stats import mannwhitneyu, rankdata
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

# Network generation
import matplotlib.cm as cm
import networkx as nx
import community as community_louvain

# TRIASSIC utilities
from raptcr.io.pipeline import ProcessingPipeline
from raptcr.io.mappers import RegexMapper

# Local project utilities
from utils.conv_clustering import (
    analyze_clusters,
    count_genus_prevalence,
    get_clusters,
    load_genus_results,
    parse_mixcr,
)

# Global settings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

In [ ]:
base_dir = Path("..").resolve()

# Discovery dataset directories
disc_data_dir = base_dir / "1_raw_data" / "0_discovery_data"
disc_raw_tcr_dir = disc_data_dir / "0_discovery_TCR"
disc_conv_dir = base_dir / "2_processed_data" / "3_discovery_convergence"
disc_conv_dir.mkdir(exist_ok=True)

# Brand et al. IBD dataset
ibd_data_dir = base_dir / "1_raw_data" / "1_Brand_data"
ibd_processed_dir = base_dir / "2_processed_data" / "1_Brand_data"
ibd_conv_dir = base_dir / "2_processed_data" / "4_Brand_convergence"
ibd_conv_dir.mkdir(exist_ok=True)

# Pu et al. CRC dataset
crc_data_dir = base_dir / "1_raw_data" / "2_Pu_data"
crc_processed_dir = base_dir / "2_processed_data" / "2_Pu_data"
crc_conv_dir = base_dir / "2_processed_data" / "5_Pu_convergence"
crc_conv_dir.mkdir(exist_ok=True)

In [ ]:
# All 36 genera included in the analysis
sorted_genera = [
    "Peptoniphilus",
    "Lactobacillus",
    "Desulfovibrio",
    "Porphyromonas",
    "Erysipelotrichaceae",
    "Actinomyces",
    "Dialister",
    "Phascolarctobacterium",
    "Oxalobacter",
    "Paraprevotella",
    "Akkermansia",
    "Parasutterella",
    "Coprococcus",
    "Odoribacter",
    "Coprobacter",
    "Bilophila",
    "Bacteroidales",
    "Sutterella",
    "Prevotella",
    "Streptococcus",
    "Ruminococcaceae",
    "Eubacterium",
    "Escherichia",
    "Clostridium",
    "Pseudoflavonifractor",
    "Anaerostipes",
    "Collinsella",
    "Bifidobacterium",
    "Ruminococcus",
    "Lachnospiraceae",
    "Roseburia",
    "Alistipes",
    "Dorea",
    "Faecalibacterium",
    "Oscillibacter",
    "Parabacteroides",
]

# 1. Discovery dataset preparation

### 1A. Get discovery TCR data

In [ ]:
# Input files
overlap_file = disc_data_dir / "Overlap_id_list.csv"
microbiome_file = disc_data_dir / "Microbiome_full_names_taxa.csv"

# Output files
alpha_out = disc_conv_dir / "TRA_dataframe.csv"
beta_out = disc_conv_dir / "TRB_dataframe.csv"
genus_mtx = disc_conv_dir / "genus_count_matrix_th1.csv"

# Process TCR repertoires with TRIASSIC
patient_id_mapper = RegexMapper(pattern=r"(HH\d+|CO\d+|PT\d+)")
repertoire_id_mapper = RegexMapper(pattern=r"[-_]([Covid|ERC].*[_-]CD[48])")

pipe = ProcessingPipeline(
    patient_id_mapping=patient_id_mapper,
    repertoire_id_mapping=repertoire_id_mapper,
)

# Use the raw TCR data
disc_data = pipe.process_dataset(disc_raw_tcr_dir)

# Add chain label (TRA/TRB) and fix IDs
disc_data["chain"] = disc_data["v_gene"].str[:3]
disc_data["patient_id"] = disc_data["patient_id"].replace(
    {"HH0781": "HH078.1", "HH0782": "HH078.2"}
)
disc_data["repertoire_id"] = disc_data["repertoire_id"].replace(
    {"ERC-CO182-CD4": "ERC_CO182_CD4", "ERC-CO182-CD8": "ERC_CO182_CD8"}
)

# Filter to patients with overlapping microbiome data
overlapping = pd.read_csv(overlap_file)
disc_data = disc_data[disc_data["patient_id"].isin(overlapping["patient_id"])]

# Split into alpha and beta repertoires
data_alpha = disc_data.query("chain == 'TRA'").reset_index(drop=True)
data_beta = disc_data.query("chain == 'TRB'").reset_index(drop=True)

# Format the V- and J-genes
for df in (data_alpha, data_beta):
    df["v_call"] = df["v_gene"] + "*01"
    df["j_call"] = df["j_gene"] + "*01"

# Save processed repertoires
data_alpha.to_csv(alpha_out, index=False)
data_beta.to_csv(beta_out, index=False)

### 1B. Get genus level microbiom count matrix

In [ ]:
# Get all patient IDs with paired TCR and microbiome data
patient_list = overlapping["patient_id"]

# Read in the microbiome count matrix
full_micro = pd.read_csv(microbiome_file)

# Select those samples that also have TCR data
full_micro_overlap = full_micro[full_micro["subject"].isin(patient_list)]

# Extract genus level and aggregate abundance at the genus level
full_micro_overlap["genus"] = full_micro_overlap["taxon_name"].str.split(" ").str[0]
matrix_genus = (
    full_micro_overlap.groupby(["genus", "subject"])["rel_abundance"]
    .sum()
    .reset_index()
)
matrix_genus = pd.pivot_table(
    matrix_genus, values="rel_abundance", index="genus", columns="subject", fill_value=0
)

# Keep only genera found in more than one subject
matrix_genus = matrix_genus[(matrix_genus != 0).sum(axis=1) > 1]

# Save genus abundance matrix
matrix_genus.T.to_csv(genus_mtx)

# 2. Brand et al. data preparation

In [ ]:
# Load the Brand et al. TCR data
ibd_data = pd.read_csv(ibd_processed_dir / "0_parsed_full_IBD_data.csv")

# Adjust metadata to fill empty values
ibd_data["Concordancy"] = ibd_data["Concordancy"].fillna("Healthy")
ibd_data.loc[ibd_data["Concordancy"] == "Healthy", "Diagnosis"] = "Healthy"

# Select the correct columns and samples
ibd_data = ibd_data[
    [
        "junction",
        "v_call",
        "junction_aa",
        "j_call",
        "patient_id",
        "Diagnosis",
        "chain_corrected",
    ]
]

twin_samples = [
    "TWIN0001",
    "TWIN0002",
    "TWIN0011",
    "TWIN0012",
    "TWIN0015",
    "TWIN0016",
    "TWIN0041",
    "TWIN0042",
    "TWIN0045",
    "TWIN0046",
    "TWIN0081",
    "TWIN0082",
    "TWIN0099",
    "TWIN0100",
    "TWIN0101",
    "TWIN0102",
]

# Split into alpha and beta dataframes
ibd_data_alpha = ibd_data[
    (ibd_data["chain_corrected"] == "TRA") & (ibd_data["patient_id"].isin(twin_samples))
]
ibd_data_beta = ibd_data[
    (ibd_data["chain_corrected"] == "TRB") & (ibd_data["patient_id"].isin(twin_samples))
]

# Save data
ibd_data_alpha.to_csv(ibd_conv_dir / "TRA_dataframe.csv", index=False)
ibd_data_beta.to_csv(ibd_conv_dir / "TRB_dataframe.csv", index=False)

In [ ]:
# Load microbiome counts
ibd_micro = pd.read_excel(ibd_data_dir / "2_microbiome_species.xlsx", index_col=0)

# Extract microbiome species level
ibd_micro.columns = ibd_micro.columns.str.extract(r"\.s__(.+)$")[0]

# Extract the genus name
ibd_micro.columns = ibd_micro.columns.str.extract(r"^([^_]+)_")[0]

# Aggregate at the genus level
ibd_micro = ibd_micro.T.groupby(level=0).sum().T

# Keep only genera present in at least 2 individuals
ibd_micro = ibd_micro.loc[:, (ibd_micro != 0).sum(axis=0) > 1]

ibd_micro.to_csv(ibd_conv_dir / "genus_count_matrix_th1.csv")

# 3. Pu et al. data preparation

In [ ]:
crc_data = pd.read_csv(crc_processed_dir / "0_Janssen_TCR.csv")

# Drop duplicate entries based on the patient and their clone ID, different cells will be kept
crc_data = crc_data.drop_duplicates(subset=["sample_tissue", "clone_id_aa_identity"])

# Split the single cell dataframe into alpha and beta sequences
crc_alpha = (
    crc_data[
        [
            "patient_id",
            "tissue",
            "sample_tissue",
            "MSS_type",
            "iCMS_type",
            "majorSTF",
            "cell_type",
            "IR_VJ_1_junction",
            "IR_VJ_1_junction_aa",
            "IR_VJ_1_v_call",
            "IR_VJ_1_j_call",
            "clone_id_aa_identity",
            "full_tcr",
        ]
    ]
    .rename(
        columns={
            "IR_VJ_1_junction": "junction",
            "IR_VJ_1_junction_aa": "junction_aa",
            "IR_VJ_1_v_call": "v_call",
            "IR_VJ_1_j_call": "j_call",
            "sample_tissue": "repertoire_id",
        }
    )
    .dropna(subset=["junction_aa"])
)
crc_beta = (
    crc_data[
        [
            "patient_id",
            "tissue",
            "sample_tissue",
            "MSS_type",
            "iCMS_type",
            "majorSTF",
            "cell_type",
            "IR_VDJ_1_junction",
            "IR_VDJ_1_junction_aa",
            "IR_VDJ_1_v_call",
            "IR_VDJ_1_j_call",
            "clone_id_aa_identity",
            "full_tcr",
        ]
    ]
    .rename(
        columns={
            "IR_VDJ_1_junction": "junction",
            "IR_VDJ_1_junction_aa": "junction_aa",
            "IR_VDJ_1_v_call": "v_call",
            "IR_VDJ_1_j_call": "j_call",
            "sample_tissue": "repertoire_id",
        }
    )
    .dropna(subset=["junction_aa"])
)


crc_alpha = parse_mixcr(crc_alpha, rename=False)
crc_alpha = crc_alpha.dropna().reset_index(drop=True)
crc_beta = parse_mixcr(crc_beta, rename=False)
crc_beta = crc_beta.dropna().reset_index(drop=True)

crc_alpha.to_csv(crc_conv_dir / "TRA_dataframe.csv")
crc_beta.to_csv(crc_conv_dir / "TRB_dataframe.csv")

In [ ]:
crc_micro = pd.read_csv(
    crc_processed_dir / "1_Micro_count_matrix_full.csv", index_col=0
)

# Extract the genera as column names
crc_micro.columns = crc_micro.columns.str.extract(r"^(.+?) ")[0]

# Sum the counts per genus, then only keep the genera that are present in at least 2 different patients
crc_micro = crc_micro.T.groupby(level=0).sum().T
crc_micro = crc_micro.loc[:, (crc_micro != 0).sum(axis=0) > 1]

crc_micro.to_csv(crc_conv_dir / "genus_count_matrix_th1.csv")

# 4. Get shared genera and calculate convergence per genus

### 4A. Get the microbiome matrices of all three datasets

Compare the genus-level count matrices and extract genera present in all 3 of the datasets

In [ ]:
# Load genus count matrices
disc_matrix = pd.read_csv(disc_conv_dir / "genus_count_matrix_th1.csv", index_col=0)
ibd_matrix = pd.read_csv(ibd_conv_dir / "genus_count_matrix_th1.csv", index_col=0)
crc_matrix = pd.read_csv(crc_conv_dir / "genus_count_matrix_th1.csv", index_col=0)

# Identify overlapping genera across datasets
disc_list, ibd_list, crc_list = map(
    set, [disc_matrix.columns, ibd_matrix.columns, crc_matrix.columns]
)
overlap_list = list(disc_list & ibd_list & crc_list)

print(
    f"ERC data: {len(disc_list)} unique genera\n"
    f"IBD data: {len(ibd_list)} unique genera\n"
    f"Janssen data: {len(crc_list)} unique genera\n"
    f"Total overlapping genera: {len(overlap_list)}"
)

# Keep only overlapping genera
for matrix, conv_dir in zip(
    [disc_matrix, ibd_matrix, crc_matrix], [disc_conv_dir, ibd_conv_dir, crc_conv_dir]
):
    matched_matrix = matrix.loc[:, matrix.columns.isin(overlap_list)]
    matched_matrix.to_csv(conv_dir / "genus_count_matrix_matched.csv")

### 4B. Calculate convergence for each of the 36 shared genera separately

Use these shared genera in the script 3A_genus_convergence.py to calculate convergence values for each genus in each dataset.

Takes +- 10 hours to run the full analysis

(This does not include Peptoniphilus since this is not present in the twins dataset)

(Also we removed Bacteroides from the list (later) since it was present in all individuals, leading to no negative group in the analysis)

### 4C. Load the convergence results for each genus and each dataset

This includes the TCR results from the analysis of all 36 genera and the TCRs for specifically Prevotella bivia and Peptoniphilus lacrimalis ("Selected" data)

In [ ]:
# Load discovery, crc, and ibd genus convergence results
disc_results_dir = (
    base_dir / "2_processed_data" / "3_discovery_convergence" / "genus_results"
)
ibd_results_dir = (
    base_dir / "2_processed_data" / "4_Brand_convergence" / "genus_results"
)
crc_results_dir = base_dir / "2_processed_data" / "5_Pu_convergence" / "genus_results"


combined_disc = load_genus_results(disc_results_dir, file_selection="full_genera")
combined_disc["dataset"] = "Discovery"
combined_disc_selected = load_genus_results(
    disc_results_dir, file_selection="selected_genera"
)
combined_disc_selected["dataset"] = "Discovery"

combined_ibd = load_genus_results(ibd_results_dir, file_selection="full_genera")
combined_ibd["dataset"] = "Brand"
combined_ibd_selected = load_genus_results(
    ibd_results_dir, file_selection="selected_genera"
)
combined_ibd_selected["dataset"] = "Brand"

combined_crc = load_genus_results(crc_results_dir, file_selection="full_genera")
combined_crc["dataset"] = "Pu"
combined_crc_selected = load_genus_results(
    crc_results_dir, file_selection="selected_genera"
)
combined_crc_selected["dataset"] = "Pu"

# Combine all datasets
combined = pd.concat([combined_disc, combined_ibd, combined_crc])
combined_selected = pd.concat(
    [combined_disc_selected, combined_ibd_selected, combined_crc_selected]
)

combined_selected["genus"] = combined_selected["genus"].str.split("_").str[0]

final_conv = pd.concat([combined, combined_selected], ignore_index=True)

final_conv["repertoire_id"] = final_conv["repertoire_id"].replace(
    {"ERC-CO182-CD4": "ERC_CO182_CD4", "ERC-CO182-CD8": "ERC_CO182_CD8"}
)

# Reorder columns
final_conv = final_conv[
    [
        "junction",
        "junction_aa",
        "v_call",
        "j_call",
        "patient_id",
        "repertoire_id",
        "dataset",
        "Diagnosis",
        "tissue",
        "MSS_type",
        "iCMS_type",
        "majorSTF",
        "cell_type",
        "clone_id_aa_identity",
        "full_tcr",
        "match_true",
        "match_false",
        "background_true",
        "background_false",
        "statistic",
        "pvalue",
        "convergence",
        "genus",
    ]
]

final_conv = final_conv.drop_duplicates()

# Save combined results
output_dir = base_dir / "2_processed_data/6_combined_convergence"
output_dir.mkdir(exist_ok=True)
final_conv.to_csv(output_dir / "0_combined_final_sign_interactions.csv", index=False)

# 5. Cluster convergent TCRs for each genus separately and identify shared TCRs

In [ ]:
final_conv = pd.read_csv(
    base_dir
    / "2_processed_data"
    / "6_combined_convergence"
    / "0_combined_final_sign_interactions.csv",
    index_col=0,
)
final_conv = final_conv[
    (final_conv["convergence"] > 2) & (final_conv["pvalue"] <= 0.05)
]

# Per genus, cluster the convergent TCRs and add cluster information to base dataframe
for i in final_conv["genus"].unique():
    print(i)
    dataframe = final_conv[final_conv["genus"] == i]
    get_clusters(
        dataframe, i, base_dir=base_dir / "2_processed_data" / "7_cluster_convergence"
    )

In [ ]:
# Explore the clustered convergent TCRs and identify shared clusters among the three dataset
results = analyze_clusters(
    genera=sorted_genera,
    cluster_dir=base_dir / "2_processed_data" / "7_cluster_convergence",
    output_dir=base_dir / "2_processed_data" / "6_combined_convergence",
    convergence_threshold=2,
    pvalue_threshold=0.05,
    top_percentile=10,
)

In [ ]:
results["adjusted_pval"]

In [ ]:
results["grouped"]

# 6. Convergence overview analysis

### Add TCR-microbiome publicity information to the dataframe

In [ ]:
# How many TCRs were there originally for the convergence calculations?
data_alpha = pd.read_csv(disc_conv_dir / "TRA_dataframe.csv")
data_beta = pd.read_csv(disc_conv_dir / "TRB_dataframe.csv")
together = pd.concat([data_alpha, data_beta])
together["disc_celltype"] = together["repertoire_id"].str.split(r"[_-]").str[2]
together["group"] = together["chain"] + "_" + together["disc_celltype"]

# Load convergence results
final_conv = pd.read_csv(
    base_dir
    / "2_processed_data"
    / "6_combined_convergence"
    / "0_combined_final_sign_interactions.csv"
)
overview_df = pd.read_csv(
    base_dir / "2_processed_data" / "6_combined_convergence" / "2_overview_df.csv"
)
disc_genus = pd.read_csv(
    base_dir
    / "2_processed_data"
    / "3_discovery_convergence"
    / "genus_count_matrix_th1.csv",
    index_col=0,
)


# Select only the discovery dataset results and divide into groups based on chain and cell type
conv_disc = final_conv[
    (final_conv["dataset"] == "Discovery")
    & (final_conv["convergence"] > 2)
    & (final_conv["pvalue"] <= 0.05)
]

conv_disc["chain"] = conv_disc["v_call"].str.extract(r"(TR[A|B])")
conv_disc["disc_celltype"] = conv_disc["repertoire_id"].str.split(r"[_-]").str[2]
conv_disc["group"] = conv_disc["chain"] + "_" + conv_disc["disc_celltype"]
conv_disc["full_tcr"] = (
    conv_disc["chain"]
    + "_"
    + conv_disc["disc_celltype"]
    + "_"
    + conv_disc["v_call"]
    + "_"
    + conv_disc["junction_aa"]
    + "_"
    + conv_disc["j_call"]
)

# Calculate the number of patients in which a VDJ TCR is linked to the same genus
tcr_patient_counts = (
    conv_disc.groupby(["junction_aa", "v_call", "j_call", "genus"])["patient_id"]
    .nunique()
    .reset_index()
    .rename(columns={"patient_id": "patient_count"})
)

# Add publicity information to the convergence data
conv_disc = conv_disc.merge(
    tcr_patient_counts, on=["junction_aa", "v_call", "j_call", "genus"], how="left"
).drop(
    columns={
        "Diagnosis",
        "tissue",
        "MSS_type",
        "iCMS_type",
        "majorSTF",
        "cell_type",
        "clone_id_aa_identity",
    }
)

### Calculate genus prevalence for the included genera

In [ ]:
# Calculate the genus prevalence per genus across the individuals in the discovery cohort
additional_info = []

for genus in sorted_genera:
    genus_present, genus_absent = count_genus_prevalence(disc_genus, genus)
    total = genus_present + genus_absent
    prevalence = genus_present / total if total > 0 else 0
    additional_info.append(
        {
            "genus": genus,
            "Genus present": genus_present,
            "Genus absent": genus_absent,
            "genus_prevalence": prevalence,
        }
    )
additional_df = pd.DataFrame(additional_info)

### Create convergence overview and compare cell type differences in discovery dataset

In [ ]:
# Create summary dataframe for mean convergence results per genus
convergence_summary = (
    conv_disc.groupby("genus")
    .agg(
        median_convergence=("convergence", "median"),
        mean_convergence=("convergence", "mean"),
        median_pvalue=("pvalue", "median"),
        mean_pvalue=("pvalue", "mean"),
        dominant_chain=("chain", lambda x: x.mode()[0] if len(x.mode()) > 0 else "NA"),
    )
    .reset_index()
)

convergence_summary["neg_log10_p"] = -np.log10(convergence_summary["mean_pvalue"])

# Add genus prevalence
convergence_summary = convergence_summary.merge(additional_df, on="genus", how="left")
convergence_summary["Genus prevalence (%)"] = convergence_summary["genus_prevalence"]
convergence_summary.to_csv(
    base_dir
    / "2_processed_data"
    / "6_combined_convergence"
    / "5_convergence_summary.csv"
)


# Create summary dataframe for chain- and cell-type results per genus
df = conv_disc.copy()

tcr_counts = (
    df.groupby(["genus", "group"])
    .agg(num_tcrs=("full_tcr", "count"))
    .unstack(fill_value=0)
)
mean_convergence = (
    df.groupby(["genus", "group"])
    .agg(mean_convergence=("convergence", "mean"))
    .unstack(fill_value=0)
)
mean_publicity = (
    df.groupby(["genus", "group"])
    .agg(mean_publicity=("patient_count", "mean"))
    .unstack(fill_value=0)
)

# Get the total number of TCRs per genus (combined across all 4 groups)
total_tcrs = df.groupby("genus").agg(total_tcrs=("full_tcr", "count"))
tcr_counts.columns = [f"{col[1]}" for col in tcr_counts.columns]
mean_convergence.columns = [
    f"{col[1]}_mean_convergence" for col in mean_convergence.columns
]
mean_publicity.columns = [f"{col[1]}_mean_publicity" for col in mean_publicity.columns]

# Create the big summary table
cell_group_summary = (
    total_tcrs.join(tcr_counts, how="left", rsuffix="_count")
    .join(mean_convergence, how="left", rsuffix="_meanconv")
    .join(mean_publicity, how="left", rsuffix="_meanpub")
)
cell_group_summary.columns = [
    "_".join(col).strip() if isinstance(col, tuple) else col
    for col in cell_group_summary.columns.values
]
cell_group_summary = cell_group_summary.reset_index()

# Take the fraction of the cells based on how abundant each group was at the start in original discovery data
for column in ["TRA_CD4", "TRA_CD8", "TRB_CD4", "TRB_CD8"]:
    cell_group_summary[f"rel_{column}"] = (
        cell_group_summary[column] / together["group"].value_counts()[column]
    ) * 100
cell_group_summary = cell_group_summary.sort_values("total_tcrs", ascending=False)
cell_group_summary.to_csv(
    base_dir
    / "2_processed_data"
    / "6_combined_convergence"
    / "6_cell_group_summary.csv"
)

### Create convergence overview for all three datasets

In [ ]:
# Cluster overview per genus
overview_df = overview_df.rename(
    columns={
        "Genus": "genus",
        "# publicity <3": "# Clusters -- not shared",
        "# publicity = 3": "# Clusters -- Shared",
        "Adjusted p-value": "Enrichement p val (adj.)",
    }
)

# Individual convergent TCRs
conv_selected = final_conv[
    (final_conv["convergence"] > 2) & (final_conv["pvalue"] <= 0.05)
]

conv_selected["full_tcr"] = (
    conv_selected["v_call"]
    + "_"
    + conv_selected["junction_aa"]
    + "_"
    + conv_selected["j_call"]
)

# Extract the number of convergent TCRs per genus for all three datasets
genus_char = (
    conv_selected.groupby(["genus", "dataset"])["full_tcr"]
    .count()
    .unstack(fill_value=0)
    .reset_index()
)

# Merge the TCR counts into the overview df
genus_char = pd.merge(genus_char, overview_df, on="genus", how="inner")
genus_char = (
    pd.merge(genus_char, additional_df, on="genus", how="inner")
    .rename(
        columns={
            "genus": "Bacterial genus",
            "Discovery": "# Convergent TCRs (Discovery)",
            "Pu": "# Convergent TCRs (Pu et al.)",
            "Brand": "# Convergent TCRs (Brand et al.)",
            "genus_prevalence": "Genus prevalence (Discovery)",
        }
    )
    .sort_values(by="# Convergent TCRs (Discovery)", ascending=False)
    .drop(columns={"Genus present", "Genus absent"})
    .reset_index(drop=True)
)

# Round numeric columns
genus_char["Enrichement p val (adj.)"] = genus_char["Enrichement p val (adj.)"].apply(
    lambda x: f"{x:.2e}"
)
genus_char["Genus prevalence (Discovery)"] = (
    genus_char["Genus prevalence (Discovery)"] * 100
).round(2).astype(str) + "%"

genus_char = genus_char.sort_values(by="# Convergent TCRs (Discovery)", ascending=False)

# Save dataframe
genus_char.to_csv(
    base_dir
    / "2_processed_data"
    / "6_combined_convergence"
    / "6_cluster_prevalence.csv"
)

### Test differences in cell group abundances for convergent T cells

In [ ]:
# Test differences in relative cell counts across the different groups for the identified convergent cells
cell_group_summary = cell_group_summary.merge(additional_df, on="genus", how="left")


def paired_ttest(a, b, label):
    t, p = stats.ttest_rel(cell_group_summary[a], cell_group_summary[b])
    print(f"\nStats for {label}:")
    print(f"T-statistic: {t:.3f}")
    print(f"P-value: {p:.3g}")


paired_ttest("rel_TRA_CD4", "rel_TRA_CD8", "TCR alpha chain (CD4 vs CD8)")
paired_ttest("rel_TRB_CD4", "rel_TRB_CD8", "TCR beta chain (CD4 vs CD8)")
paired_ttest("rel_TRA_CD4", "rel_TRB_CD4", "CD4 TRA vs TRB")
paired_ttest("rel_TRA_CD8", "rel_TRB_CD8", "CD8 TRA vs TRB")

# 7. Convergence covariate correction

In [ ]:
# Metadata including potential covariates for convergence correction
meta = pd.read_excel(
    base_dir / "1_raw_data" / "0_discovery_data" / "AIRRWAS_metadata.xlsx"
)
meta["patient_id"] = meta["Subjectnr"].str.split(" ").str[1]

# Input error fix
meta.loc[meta["Subjectnr"] == "Covid PT026", "Weight kg"] = 79
meta.loc[meta["Subjectnr"] == "Covid PT026", "BMI"] = 27.335

# Microbiome matrix per individual
micro = pd.read_csv(
    base_dir
    / "2_processed_data"
    / "3_discovery_convergence"
    / "genus_count_matrix_th1.csv",
    index_col=0,
)
# Prevalence = fraction of patients with abundance > 0
prevalence = (micro > 0).sum(axis=0) / micro.shape[0]
prevalence = prevalence.reset_index()
prevalence.columns = ["genus", "prevalence"]

# Files for covariate convergence correction
genus_results_erc = (
    base_dir / "2_processed_data" / "3_discovery_convergence" / "genus_results"
)
conv_correction_dir = base_dir / "2_processed_data" / "9_conv_correction"
conv_correction_dir.mkdir(exist_ok=True)

In [ ]:
######################################
### Calculate residual convergence ###
###        +- 30-40 min            ###
######################################

genus_files = defaultdict(list)

# Find all convergence results files for the discovery dataset
pattern = r"^(?!X_)(.+?)_(alpha|beta)_results\.csv$"
for file in os.listdir(genus_results_erc):
    match = re.match(pattern, file)
    if match:
        genus = match.group(1)
        chain = match.group(2)
        genus_files[genus].append((file, chain))


# Manual additions for special cases
# List of tuples: (filename, genus_name, chain)
manual_files = [
    ("X_Peptoniphilus_alpha_results.csv", "Peptoniphilus", "alpha"),
    ("X_Peptoniphilus_beta_results.csv", "Peptoniphilus", "beta"),
]

for fname, genus_name, chain in manual_files:
    file_path = os.path.join(genus_results_erc, fname)
    if os.path.exists(file_path):
        genus_files[genus_name].append((fname, chain))
    else:
        print(f"Warning: manual file {fname} not found in directory")


# Run loop to apply linear model to all convergence values per genus and chain
all_results = []
tail_dfs = []
covariate_effects_list = []

# Looping over the different genera
for genus, files in genus_files.items():
    print(f"Processing {genus}")

    dfs = []
    # Load convergence file
    for file, chain in files:
        df = pd.read_csv(os.path.join(genus_results_erc, file), index_col=0)
        df["chain"] = chain
        dfs.append(df)

    df = pd.concat(dfs, ignore_index=True)
    df = df.merge(meta, on="patient_id", how="left")

    # Full-distribution statistics
    conv_mean = df["convergence"].mean()
    conv_sd = df["convergence"].std()
    age_sd = df["Age"].std()
    bmi_sd = df["BMI"].std()

    # Calculate residualized convergence with linear model
    model = smf.ols(
        "convergence ~ Age + BMI + C(Gender)", data=df  # + C(chain) + C(Group)
    ).fit()

    df["conv_residual"] = model.resid
    df["delta_conv"] = df["convergence"] - df["conv_residual"]

    # Standardized effects
    age_std = model.params["Age"] * age_sd / conv_sd
    bmi_std = model.params["BMI"] * bmi_sd / conv_sd

    # Categorical effects
    cat_effects = {
        k: v / conv_sd for k, v in model.params.items() if k.startswith("C(")
    }

    max_cat_effect = max(np.abs(list(cat_effects.values())))

    # Statistics
    r2_cov = model.rsquared
    rho, rho_p = stats.spearmanr(df["convergence"], df["conv_residual"])

    # Store genus-level summary
    all_results.append(
        {
            "genus": genus,
            "full_conv_mean": conv_mean,
            "full_conv_sd": conv_sd,
            "Age_std_effect": age_std,
            "BMI_std_effect": bmi_std,
            "max_categorical_effect": max_cat_effect,
            "median_delta_conv": df["delta_conv"].median(),
            "mean_delta_conv": df["delta_conv"].mean(),
            "R2_covariates": r2_cov,
            "Spearman_rho": rho,
            "Spearman_p": rho_p,
        }
    )

    # Store covariate effects per genus
    covariate_effects_list.append(
        {"genus": genus, "Age": age_std, "BMI": bmi_std, **cat_effects, "R2": r2_cov}
    )

    # Select the tail of the distribution based on the residuals and original convergence values, and save those for later analysis
    df_tail = df[
        ((df["conv_residual"] > 2) | (df["convergence"] > 2)) & (df["pvalue"] <= 0.05)
    ].copy()

    df_tail["genus"] = genus
    tail_dfs.append(df_tail)

# Save tail results
combined_tail_df = pd.concat(tail_dfs, ignore_index=True)
combined_tail_df.to_csv(conv_correction_dir / "all_tails_combined.csv", index=False)

# Save genus-level summary
results_df = pd.DataFrame(all_results)
results_df.to_csv(conv_correction_dir / "covariate_summary.csv", index=False)

# Save covariate effects table
covariate_effects_df = pd.DataFrame(covariate_effects_list)
covariate_effects_df.to_csv(
    conv_correction_dir / "covariate_effects_detailed.csv", index=False
)

# 8. Network generation

In [ ]:
# Load all significantly convergent results and select only the discovery dataset results
final_conv = pd.read_csv(
    base_dir
    / "2_processed_data"
    / "6_combined_convergence"
    / "0_combined_final_sign_interactions.csv",
    index_col=0,
)

conv_disc = final_conv[
    (final_conv["dataset"] == "Discovery")
    & (final_conv["convergence"] > 2)
    & (final_conv["pvalue"] <= 0.05)
]
conv_disc["chain"] = conv_disc["v_call"].str.extract(r"(TR[A|B])")

# Calculate how in how many patients a VDJ TCR is linked to the same genus
tcr_patient_counts = (
    final_conv.groupby(["junction_aa", "v_call", "j_call"])["patient_id"]
    .nunique()
    .reset_index()
    .rename(columns={"patient_id": "patient_count"})
)

conv_disc = conv_disc.merge(
    tcr_patient_counts, on=["junction_aa", "v_call", "j_call"], how="left"
)

# Create the network dataframe for public TCRs, so CDR3s present in 4 indivuals or more
network_df = conv_disc[
    [
        "junction_aa",
        "v_call",
        "j_call",
        "chain",
        "genus",
        "convergence",
        "patient_count",
    ]
].drop_duplicates()
network_df = network_df[network_df["patient_count"] > 4]

# Add node IDs to each unique TCR and genus
new_df = pd.concat([network_df["junction_aa"], network_df["genus"]])
unique = len(new_df.unique())
df = pd.DataFrame()
df["node_id"] = range(1, unique + 1, 1)
df["node_name"] = new_df.unique()
dictionary = df.set_index("node_name").to_dict()["node_id"]

network_df["node_id_var1"] = network_df["junction_aa"]
network_df["node_id_var2"] = network_df["genus"]
network_df = network_df.replace({"node_id_var1": dictionary})
network_df = network_df.replace({"node_id_var2": dictionary})

network_df.to_csv(
    base_dir / "2_processed_data" / "3_discovery_convergence" / "Network_dataframe.csv"
)
network_df

# 9. Cross-cohort convergence

In [ ]:
shared_clusters = pd.read_csv(
    base_dir / "2_processed_data" / "6_combined_convergence" / "4_red_cluster_TCR.csv"
)

shared_clusters["genus_clus"] = (
    shared_clusters["genus"].astype(str) + "_" + shared_clusters["cluster"].astype(str)
)

shared_clusters["full_vdj"] = (
    shared_clusters["v_call"]
    + "_"
    + shared_clusters["junction_aa"]
    + "_"
    + shared_clusters["j_call"]
)

print(
    "Number of red clusters across all genera:", shared_clusters["genus_clus"].nunique()
)
print("Number of unique TCRs in red clusters:", shared_clusters["full_vdj"].nunique())
print("Number of genera with red clusters:", shared_clusters["genus"].nunique())

Suppl_data_table = shared_clusters[
    [
        "junction_aa",
        "v_call",
        "j_call",
        "patient_id",
        "dataset",
        "cluster",
        "size",
        "motif",
        "genus_clus",
        "genus",
        "pvalue",
        "convergence",
    ]
]

Suppl_data_table["dataset"] = Suppl_data_table["dataset"].replace(
    {"ERC": "Discovery", "Janssen": "Ting et al.", "Twins": "Brand et al."}
)

Suppl_data_table.to_csv(base_dir / "3_figures" / "Tables" / "Shared_clusters_suppl.csv")
Suppl_data_table

# 10. T cell subtype preference in convergent TCRs

### Calculate cell type distirbutions for the three datasets

In [ ]:
subset_dir = base_dir / "2_processed_data" / "8_T_cell_subsets"
subset_dir.mkdir(exist_ok=True)

### Discovery dataset

In [ ]:
# Original unfiltered data before convergence analysis (=All TCRs used for the convergence analysis)
data_alpha = pd.read_csv(
    base_dir / "2_processed_data" / "3_discovery_convergence" / "TRA_dataframe.csv"
)
data_beta = pd.read_csv(
    base_dir / "2_processed_data" / "3_discovery_convergence" / "TRB_dataframe.csv"
)

together = pd.concat([data_alpha, data_beta])
together["erc_cell"] = together["repertoire_id"].str.split(r"[_-]").str[2]
together["group"] = together["chain"] + "_" + together["erc_cell"]


# All TCRs that were found to be significantly convergent
final_conv = pd.read_csv(
    base_dir
    / "2_processed_data"
    / "6_combined_convergence"
    / "0_combined_final_sign_interactions.csv"
)
final_conv = final_conv[final_conv["dataset"] == "Discovery"]


# Compare the T cell subtypes included in the convergent groups versus at the start
full_tcr_info = pd.merge(
    together,
    final_conv[
        [
            "junction",
            "junction_aa",
            "v_call",
            "j_call",
            "patient_id",
            "statistic",
            "pvalue",
            "convergence",
            "genus",
        ]
    ],
    how="left",
)
full_tcr_info = full_tcr_info[
    [
        "junction",
        "junction_aa",
        "v_call",
        "j_call",
        "patient_id",
        "erc_cell",
        "statistic",
        "pvalue",
        "convergence",
        "genus",
        "group",
    ]
].drop_duplicates()


full_tcr_info["convergent"] = np.where(
    (full_tcr_info["convergence"] > 2) & (full_tcr_info["pvalue"] < 0.05), "Yes", "No"
)

before_df = full_tcr_info.drop_duplicates(
    subset=["junction", "junction_aa", "v_call", "j_call", "patient_id", "erc_cell"]
).drop(columns={"statistic", "pvalue", "convergence", "genus", "convergent"})

keys = ["junction", "junction_aa", "v_call", "j_call", "patient_id", "erc_cell"]
before_hash = before_df[keys].apply(lambda x: hash(tuple(x)), axis=1).to_numpy()

before_df.to_csv(subset_dir / "1_before_disc_data.csv")
full_tcr_info.to_csv(subset_dir / "1_full_info_disc_data.csv")

# Pu et al. dataset

In [ ]:
# Original unfiltered data before convergence analysis (=All TCRs used for the convergence analysis)
data_alpha = pd.read_csv(
    base_dir / "2_processed_data" / "5_Pu_convergence" / "TRA_dataframe.csv"
)
data_beta = pd.read_csv(
    base_dir / "2_processed_data" / "5_Pu_convergence" / "TRB_dataframe.csv"
)
together = pd.concat([data_alpha, data_beta])

# All TCRs that were found to be significantly convergent
final_conv = pd.read_csv(
    base_dir
    / "2_processed_data"
    / "6_combined_convergence"
    / "0_combined_final_sign_interactions.csv"
)
final_conv = final_conv[final_conv["dataset"] == "Pu"]

# Compare the T cell subtypes included in the convergent groups versus at the start
full_tcr_info = pd.merge(
    together,
    final_conv[
        [
            "junction",
            "junction_aa",
            "v_call",
            "j_call",
            "patient_id",
            "statistic",
            "pvalue",
            "convergence",
            "genus",
        ]
    ],
    how="left",
)
full_tcr_info = full_tcr_info[
    [
        "junction",
        "junction_aa",
        "v_call",
        "j_call",
        "patient_id",
        "repertoire_id",
        "cell_type",
        "statistic",
        "pvalue",
        "convergence",
        "genus",
    ]
].drop_duplicates()


full_tcr_info["convergent"] = np.where(
    (full_tcr_info["convergence"] > 2) & (full_tcr_info["pvalue"] <= 0.05), "Yes", "No"
)

before_df = full_tcr_info.drop_duplicates(
    subset=["junction", "junction_aa", "v_call", "j_call", "patient_id", "cell_type"]
).drop(columns={"statistic", "pvalue", "convergence", "genus", "convergent"})

keys = ["junction", "junction_aa", "v_call", "j_call", "patient_id", "cell_type"]
before_hash = before_df[keys].apply(lambda x: hash(tuple(x)), axis=1).to_numpy()

before_df.to_csv(subset_dir / "2_before_Pu_data.csv")
full_tcr_info.to_csv(subset_dir / "2_full_info_Pu_data.csv")

### Brand et al. dataset

In [ ]:
# Original unfiltered data before convergence analysis (=All TCRs used for the convergence analysis)
data = pd.read_csv(
    base_dir / "2_processed_data" / "1_Brand_data" / "0_parsed_full_IBD_data.csv"
)

data["Concordancy"] = data["Concordancy"].fillna("Healthy")
data.loc[data["Concordancy"] == "Healthy", "Diagnosis"] = "Healthy"

twin_samples = [
    "TWIN0001",
    "TWIN0002",
    "TWIN0011",
    "TWIN0012",
    "TWIN0015",
    "TWIN0016",
    "TWIN0041",
    "TWIN0042",
    "TWIN0045",
    "TWIN0046",
    "TWIN0081",
    "TWIN0082",
    "TWIN0099",
    "TWIN0100",
    "TWIN0101",
    "TWIN0102",
]

data = data[data["patient_id"].isin(twin_samples)]
together = data[
    ["junction", "junction_aa", "v_call", "j_call", "patient_id", "cell_type"]
]


# All TCRs that were found to be significantly convergent
final_conv = pd.read_csv(
    base_dir
    / "2_processed_data"
    / "6_combined_convergence"
    / "0_combined_final_sign_interactions.csv"
)
final_conv = final_conv[final_conv["dataset"] == "Brand"]


# Compare the T cell subtypes included in the convergent groups versus at the start
full_tcr_info = pd.merge(
    together,
    final_conv[
        [
            "junction",
            "junction_aa",
            "v_call",
            "j_call",
            "patient_id",
            "statistic",
            "pvalue",
            "convergence",
            "genus",
        ]
    ],
    how="left",
)
full_tcr_info = full_tcr_info[
    [
        "junction",
        "junction_aa",
        "v_call",
        "j_call",
        "patient_id",
        "cell_type",
        "statistic",
        "pvalue",
        "convergence",
        "genus",
    ]
].drop_duplicates()

full_tcr_info["convergent"] = np.where(
    (full_tcr_info["convergence"] > 2) & (full_tcr_info["pvalue"] <= 0.05), "Yes", "No"
)

before_df = full_tcr_info.drop_duplicates(
    subset=["junction", "junction_aa", "v_call", "j_call", "patient_id", "cell_type"]
).drop(columns={"statistic", "pvalue", "convergence", "genus", "convergent"})

keys = ["junction", "junction_aa", "v_call", "j_call", "patient_id", "cell_type"]
before_hash = before_df[keys].apply(lambda x: hash(tuple(x)), axis=1).to_numpy()


before_df.to_csv(subset_dir / "3_before_Brand_data.csv")
full_tcr_info.to_csv(subset_dir / "3_full_info_Brand_data.csv")